In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"H:\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: H:\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_DAYS = 90  # days of lastObserved activity to include
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "Host", "URL"
]
OWNER_NAMES = [
    'Google Threat Intelligence'
]
RESULT_PAGE_SIZE = 5_000  # benchmarked stable: ~310 rows/sec with the same fields
MAX_TOTAL_RESULTS = 5_000  # hard cap on records returned by this pull

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'DateAdded >= "{start}" AND '
    'observationCount = 0 AND '
    'rating >= 3 AND '
    'confidence >= 50 AND '
    'source IS NOT NULL'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while result_start < MAX_TOTAL_RESULTS:
    current_limit = min(result_limit, MAX_TOTAL_RESULTS - result_start)
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={current_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += len(data_items)

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    #observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
    # Keep rows where top-level rating >= 3 OR coalesced threatAssessRating >= 3, and
    # (coalesced TA confidence >= 50 OR top-level confidence >= 50).
    # Coalesce flat vs nested threatAssess columns; keep top-level rating/confidence separate for OR.
    _rating_cols = ("threatAssessRating", "threatAssess.threatAssessRating", "rating")
    _confidence_cols = ("threatAssessConfidence", "threatAssess.threatAssessConfidence")

    def _first_non_null_numeric(df, ordered_cols):
        present = [c for c in ordered_cols if c in df.columns]
        if not present:
            return None
        out = pd.to_numeric(df[present[0]], errors="coerce")
        for c in present[1:]:
            s = pd.to_numeric(df[c], errors="coerce")
            out = out.mask(out.isna(), s)
        return out

    _tar = _first_non_null_numeric(observed_src, _rating_cols)
    _tc = _first_non_null_numeric(observed_src, _confidence_cols)
    if _tar is None or _tc is None:
        raise KeyError(
            f"Could not resolve Threat Assess columns. Tried rating={_rating_cols}, "
            f"confidence={_confidence_cols}. Columns: {list(observed_src.columns)}"
        )
    if "rating" in observed_src.columns:
        _r = pd.to_numeric(observed_src["rating"], errors="coerce")
    else:
        _r = pd.Series(float("nan"), index=observed_src.index, dtype=float)

    if "confidence" in observed_src.columns:
        _c = pd.to_numeric(observed_src["confidence"], errors="coerce")
    else:
        _c = pd.Series(float("nan"), index=observed_src.index, dtype=float)

    _pre_ta = len(observed_src)
    # Use >= 50 so a boundary value of 50.0 is included (strict > 50 dropped those rows).
    _pass_rating_band = (_tar >= 3) | (_r >= 3)
    _pass_confidence_band = (_tc >= 50) | (_c >= 50)
    observed_src = observed_src[_pass_rating_band & _pass_confidence_band].copy()
    display(
        f"Threat assess filter ((rating>=3 OR threatAssessRating>=3), confidence>=50) coalescing {_rating_cols} / {_confidence_cols}: "
        f"{_pre_ta} -> {len(observed_src)} rows."
    )
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)